<a href="https://colab.research.google.com/github/avinash-tiwary/ePic/blob/main/notebooks/05_3D_PIC_Plasma_Expansion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05: 3D-3V Spherical Plasma Expansion & Coulomb Explosion

## 1. Overview
This notebook demonstrates full 3D-3V Particle-In-Cell modeling of a localized plasma cloud expanding into vacuum under self-consistent 3D electrostatic fields.


In [ ]:
# ==============================================================
# Google Colab Setup & Package Installation
# ==============================================================
import sys
if 'google.colab' in sys.modules:
    print('Running in Google Colab. Installing ePic...')
    !git clone https://github.com/avinash-tiwary/ePic.git
    %cd ePic
    !pip install -e .
else:
    print('Running locally. Verifying ePic installation...')
    import epic
    print(f'ePic version {epic.__version__} loaded successfully!')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from epic.solvers.pic3d import PIC3DSolver

Nx, Ny, Nz = 24, 24, 24
Lx, Ly, Lz = 16.0, 16.0, 16.0
N_part = 30000
dt = 0.05
t_end = 5.0

weight = (1.0 * Lx * Ly * Lz) / N_part
solver = PIC3DSolver(Nx=Nx, Ny=Ny, Nz=Nz, Lx=Lx, Ly=Ly, Lz=Lz, dt=dt)

np.random.seed(42)
center = np.array([Lx/2.0, Ly/2.0, Lz/2.0])
pos_x = np.mod(np.random.normal(center[0], 2.0, N_part), Lx)
pos_y = np.mod(np.random.normal(center[1], 2.0, N_part), Ly)
pos_z = np.mod(np.random.normal(center[2], 2.0, N_part), Lz)
vel = np.zeros((N_part, 3))

solver.add_species("plasma", q=-weight, m=weight, pos_x=pos_x, pos_y=pos_y, pos_z=pos_z, vel=vel)
solver.initialize()

print("Simulating 3D Plasma Expansion...")
solver.run(t_end=t_end)

# Midplane slice
mid_z = Nz // 2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=110)
im = ax1.imshow(solver.rho[mid_z, :, :], extent=[0, Lx, 0, Ly], origin="lower", cmap="viridis")
plt.colorbar(im, ax=ax1, label=r"Density $ho(x, y, z_{mid})$")
ax1.set_xlabel("X")
ax1.set_ylabel("Y")
ax1.set_title("3D Midplane Charge Density Slice")

sp = solver.species[0]
sub = slice(None, None, 15)
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(sp.x[sub], sp.y[sub], sp.z[sub], s=1.0, c=np.linalg.norm(sp.vel[sub], axis=1), cmap="plasma")
ax2.set_xlabel("X")
ax2.set_ylabel("Y")
ax2.set_zlabel("Z")
ax2.set_title("3D Particle Spatial Distribution")
plt.tight_layout()
plt.show()


## 2. Animated 3D Coulomb Explosion Movie

<p align="center"><img src="../docs/animations/plasma_expansion_3d.gif" width="70%" alt="3D Expansion Movie"/></p>